# EVA Cesar · 03. Visualización y resultados descriptivos

Programación en Lenguajes Estadísticos · UNAL

## Montaje de Drive y ubicación del proyecto

Ejecutar desde la raíz del proyecto o indicar `EVA_RAIZ`. Correr los cuadernos en orden 01, 02 y 03.

In [ ]:
from pathlib import Path
import os

ruta_local = Path(os.environ.get("EVA_RAIZ", ".")).resolve()
try:
    from google.colab import drive
except ImportError:
    raiz = ruta_local
else:
    drive.mount("/content/drive")
    rutas = [ruta_local, Path("/content/drive/MyDrive/proyecto_eva_cesar"),
             Path("/content/drive/Shareddrives/Programacion en lenguajes estadisticos/proyecto_eva_cesar")]
    raiz = next((p for p in rutas if (p / "datos/datos_crudos").is_dir()), rutas[1])
if not (raiz / "datos/datos_crudos").is_dir():
    raise FileNotFoundError(f"Indique la carpeta raíz de EVA con EVA_RAIZ. Ruta revisada: {raiz}")
docs = raiz / "documentos"
limpios = raiz / "datos/datos_limpios"
figuras = docs / "figuras"
for carpeta in (docs, limpios, figuras):
    carpeta.mkdir(parents=True, exist_ok=True)


## Tablas anuales de rendimiento

Cargamos las versiones completa y depurada para comparar la variabilidad por grupos.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

def leer(sufijo):
    return pd.read_csv(limpios / f"eva_cesar_{sufijo}_py.csv", dtype={"cod_mun": str})

anual = leer("anual")
depurada = leer("anual_depurada")
bases = {"completa": anual, "depurada": depurada}
cultivos = sorted(anual["cultivo"].unique())

## Cobertura y comparación municipal dentro del mismo cultivo y año

In [ ]:
con_rend = depurada.dropna(subset=["rendimiento_t_ha"]).copy()
centro = con_rend.groupby(["cultivo", "ano"])["rendimiento_t_ha"].transform("median")
con_rend["desviacion"] = con_rend["rendimiento_t_ha"] - centro
territorio = con_rend.groupby(["cod_mun", "municipio", "cultivo"], as_index=False).agg(
    desviacion=("desviacion", "median"), rendimiento_t_ha=("rendimiento_t_ha", "median"), n_anos=("ano", "size"))
trayectorias = anual.groupby(["cultivo", "ano"], as_index=False).agg(
    registros=("ano", "size"), rendimiento_valido=("rendimiento_t_ha", "count"),
    area_cosechada_ha=("area_cosechada_ha", "sum"), produccion_t=("produccion_t", "sum"))
trayectorias["rendimiento_ponderado"] = trayectorias["produccion_t"] / trayectorias["area_cosechada_ha"].replace(0, np.nan)
for nombre, tabla in {"comparacion_territorial": territorio,
                      "cobertura_anual": trayectorias}.items():
    tabla.to_csv(docs / f"e3_Python_{nombre}.csv", index=False)

## Descomposición descriptiva de la variabilidad

Comparamos las sumas de cuadrados entre grupos con las sumas de cuadrados totales. Para evaluar diferencias dentro de un cultivo usamos su propia media y sus propios quintiles de área. Los porcentajes reflejan una variable a la vez y pueden solaparse.

In [ ]:
def fraccion_entre(y, centro, referencia):
    total = ((y - referencia) ** 2).sum()
    return 100 * ((centro - referencia) ** 2).sum() / total if total > 0 else np.nan

def quintiles_area(x):
    cortes = np.unique(x.quantile([.2, .4, .6, .8], interpolation="linear"))
    return pd.cut(x, np.r_[-np.inf, cortes, np.inf], labels=False, right=True)

def descomponer(datos, base):
    d = datos.loc[np.isfinite(datos["rendimiento_t_ha"]) & datos["rendimiento_t_ha"].gt(0)]
    grupos = {"cultivo": d["cultivo"], "municipio": d["cod_mun"], "ano": d["ano"],
              "area_sembrada": quintiles_area(d["area_sembrada_ha"])}
    internos = {**grupos, "area_sembrada": d.groupby("cultivo")["area_sembrada_ha"].transform(quintiles_area)}
    salida = []
    for escala, y in {"t_ha": d["rendimiento_t_ha"], "log_t_ha": np.log(d["rendimiento_t_ha"])}.items():
        cultivo = y.groupby(d["cultivo"]).transform("mean")
        for variable, grupo in grupos.items():
            pct_total = fraccion_entre(y, y.groupby(grupo).transform("mean"), y.mean())
            pct_dentro = np.nan
            if variable != "cultivo":
                medias = y.groupby([d["cultivo"], internos[variable]]).transform("mean")
                pct_dentro = fraccion_entre(y, medias, cultivo)
            salida.append(dict(base=base, escala=escala, variable=variable, n=len(d),
                               pct_total=pct_total, pct_dentro_cultivo=pct_dentro))
    return pd.DataFrame(salida)

variabilidad = pd.concat([descomponer(d, base) for base, d in bases.items()], ignore_index=True)
variabilidad.to_csv(docs / "e3_Python_variabilidad.csv", index=False)
display(variabilidad)

## Distribuciones, trayectorias y variabilidad

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", font="DejaVu Sans",
              rc={"axes.edgecolor": "#D4CADE", "axes.labelcolor": "#453B52",
                  "grid.color": "#EAE3F0", "text.color": "#362846",
                  "xtick.color": "#453B52", "ytick.color": "#453B52"})
violeta, ocre = "#6D45A6", "#AC6330"
etiquetas = {"ALGODON": "Algodón", "ARROZ": "Arroz", "CACAO": "Cacao", "CAFE": "Café",
             "CAÑA PANELERA": "Caña panelera", "FRIJOL": "Fríjol", "MAIZ": "Maíz",
             "PALMA DE ACEITE": "Palma de aceite", "PLATANO": "Plátano", "YUCA": "Yuca"}
etiquetas = {k: etiquetas.get(k, k.title()) for k in cultivos}

def guardar(fig, nombre):
    fig.savefig(figuras / f"eva_Py_{nombre}.png", dpi=190, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

puntos = anual.dropna(subset=["rendimiento_t_ha"]).copy()
puntos["nombre"] = puntos["cultivo"].map(etiquetas)
orden = puntos.groupby("nombre")["rendimiento_t_ha"].median().sort_values().index.to_list()
fig, ax = plt.subplots(figsize=(9.3, 5.6), layout="constrained")
sns.boxplot(data=puntos, x="rendimiento_t_ha", y="nombre", order=orden, color="#C7AFE6",
            showfliers=False, width=.58, linewidth=1.15, ax=ax,
            log_scale=(10, False) if puntos["rendimiento_t_ha"].gt(0).all() else None)
extremos = puntos.loc[puntos["atipico_severo"]]
# Separa las coincidencias en el eje del cultivo y conserva sus rendimientos.
coincidencias = extremos.groupby(["cultivo", "rendimiento_t_ha"])["rendimiento_t_ha"]
desplazamiento = coincidencias.transform(
    lambda grupo: np.linspace(-.25, .25, len(grupo)) if len(grupo) > 1 else 0.
)
posicion_cultivo = extremos["nombre"].map({cultivo: i for i, cultivo in enumerate(orden)})
ax.scatter(extremos["rendimiento_t_ha"], posicion_cultivo + desplazamiento,
           color=ocre, s=30, zorder=4, label="Fuera de las cercas de 3 IQR")
ax.legend(loc="upper right", frameon=False)
if puntos["rendimiento_t_ha"].gt(0).all():
    ax.set(xscale="log", xticks=[.25, .5, 1, 2, 4, 8, 16])
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%g"))
ax.set(xlabel="Rendimiento (t/ha)", ylabel="")
ax.set_title("Rendimiento anual por cultivo", loc="left", fontweight="bold", pad=13)
ax.grid(axis="y", visible=False)
guardar(fig, "02_cajas")

def paneles(titulo, subtitulo, etiqueta_x, etiqueta_y):
    fig, axs = plt.subplots(2, 5, figsize=(12, 5.4))
    fig.subplots_adjust(left=.075, right=.98, bottom=.13, top=.77, wspace=.42, hspace=.53)
    fig.suptitle(titulo, x=.075, y=.98, ha="left", fontsize=16, fontweight="bold")
    fig.text(.075, .898, subtitulo, color="#655576", fontsize=10)
    fig.supxlabel(etiqueta_x, y=.015, fontsize=10)
    fig.supylabel(etiqueta_y, x=.018, fontsize=10)
    for cultivo, ax in zip(cultivos, axs.flat):
        ax.set_title(etiquetas[cultivo], loc="left", fontsize=10, fontweight="bold")
        ax.tick_params(labelsize=8)
    return fig, axs.flat

fig, ejes = paneles("Formas del rendimiento por cultivo", "Escalas independientes por cultivo.",
                     "Rendimiento (t/ha)", "Municipios-año")
for k, eje in zip(cultivos, ejes):
    d = puntos.loc[puntos["cultivo"].eq(k)]
    eje.hist(d["rendimiento_t_ha"], bins=18, color=violeta, edgecolor="white")
guardar(fig, "03_distribuciones")

fig, ejes = paneles("Evolución del rendimiento departamental",
                     "Producción total / área cosechada total. Escalas verticales por cultivo.",
                     "Año", "Rendimiento (t/ha)")
for k, eje in zip(cultivos, ejes):
    d = trayectorias.loc[trayectorias["cultivo"].eq(k)].set_index("ano").reindex(range(2007, 2019))
    eje.plot(d.index, d["rendimiento_ponderado"], color=violeta, marker="o", markersize=3.2, lw=1.6)
    eje.set_xticks([2007, 2012, 2018])
guardar(fig, "04_trayectorias")

# Dos denominadores distintos. Las barras de cada panel pueden solaparse.
medidas = variabilidad.loc[variabilidad["base"].eq("depurada") & variabilidad["escala"].eq("t_ha")]
nombres = {"cultivo": "Cultivo", "municipio": "Municipio", "ano": "Año", "area_sembrada": "Área sembrada"}
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.35), layout="constrained")
for ax, columna, titulo, limite in zip(
    ejes, ["pct_total", "pct_dentro_cultivo"],
    ["Respecto a la variabilidad total", "Respecto a la variabilidad dentro del cultivo"], [100, 45]
):
    datos = medidas.dropna(subset=[columna]).sort_values(columna)
    barras = ax.barh(datos["variable"].map(nombres), datos[columna], color=violeta)
    etiquetas_pct = [f"{v:.1f}".replace(".", ",") + " %" for v in datos[columna]]
    ax.bar_label(barras, labels=etiquetas_pct, padding=3, fontsize=9)
    ax.set(title=titulo, xlim=(0, limite), xlabel="Porcentaje")
    ax.grid(axis="y", visible=False)
    ax.title.set_fontsize(10)
guardar(fig, "10_variabilidad")

## Mapas de calor de cobertura y territorio

El mapa de cobertura muestra en cuántos municipios hay rendimiento definido por cultivo y año.
En el mapa territorial se compara cada observación con la mediana del mismo cultivo y año.

In [ ]:
paleta_desviaciones = sns.diverging_palette(35, 275, s=75, l=45, center="light", as_cmap=True)
grilla = trayectorias.pivot(index="cultivo", columns="ano", values="rendimiento_valido").fillna(0).rename(index=etiquetas)
fig, ax = plt.subplots(figsize=(10.1, 4.8), layout="constrained")
sns.heatmap(grilla, cmap=sns.light_palette(violeta, as_cmap=True), vmin=0, vmax=25, annot=True,
            fmt=".0f", linewidths=1, linecolor="white", cbar_kws={"label": "Municipios"}, ax=ax)
ax.set(xlabel="Año", ylabel="")
ax.set_title("Municipios con rendimiento anual definido", loc="left", fontweight="bold", pad=14)
guardar(fig, "06_cobertura")

# Se muestran desviaciones en porcentaje para que la escala de color sea legible entre cultivos.
relativo = con_rend.assign(relativo=100 * con_rend["desviacion"] / centro.where(centro.ne(0)))
grilla = relativo.pivot_table(index="municipio", columns="cultivo", values="relativo", aggfunc="median")
grilla = grilla.rename(columns=etiquetas)
fig, ax = plt.subplots(figsize=(10, 8.5), layout="constrained")
sns.heatmap(grilla, cmap=paleta_desviaciones, center=0, vmin=-40, vmax=40, linewidths=.5,
            cbar_kws={"label": "Desviación mediana (%)", "extend": "both"}, ax=ax)
ax.set(xlabel="", ylabel="")
ax.set_title("Rendimiento municipal frente a la mediana del mismo cultivo y año",
             loc="left", fontweight="bold", fontsize=13, pad=13)
ax.tick_params(axis="x", rotation=28, labelsize=9)
ax.tick_params(axis="y", labelsize=8)
guardar(fig, "07_territorio")

## Exploración interactiva con Plotly

El HTML usa el archivo Plotly.js incluido en la carpeta de figuras. Abre la salida en el navegador local para cambiar el cultivo y consultar cada municipio.

In [ ]:
# Plotly.js genera un HTML portable desde Colab y permite, por defecto, consultar y ocultar series.
trazas = []
colores_series = ["#53308A", "#8054B9", "#A983CF", "#301B52", "#6D45A6", "#9174AC"]
tipos_linea = ["solid", "dot", "dash", "dashdot"]
for cultivo in cultivos:
    for i, (_, serie) in enumerate(anual.loc[anual["cultivo"].eq(cultivo)].groupby("cod_mun")):
        serie = serie.sort_values("ano")
        valores = serie.set_index("ano")["rendimiento_t_ha"].reindex(range(2007, 2019))
        trazas.append(dict(type="scatter", mode="lines+markers", name=serie["municipio"].iloc[0],
                           x=list(range(2007, 2019)), y=[float(v) if pd.notna(v) else None for v in valores],
                           line=dict(color=colores_series[i % len(colores_series)],
                                     dash=tipos_linea[i // len(colores_series) % len(tipos_linea)]),
                           connectgaps=False, visible=cultivo == cultivos[0], meta=cultivo,
                           hovertemplate="%{fullData.name}<br>%{x}: %{y:.2f} t/ha<extra></extra>"))
botones = [dict(label=etiquetas[cultivo], method="update",
                args=[dict(visible=[t["meta"] == cultivo for t in trazas]),
                      dict(title="Rendimiento municipal · " + etiquetas[cultivo])]) for cultivo in cultivos]
diseno = dict(title="Rendimiento municipal · " + etiquetas[cultivos[0]], paper_bgcolor="white", plot_bgcolor="white",
              xaxis=dict(title=dict(text="Año"), dtick=1), yaxis=dict(title=dict(text="Rendimiento (t/ha)")),
              margin=dict(t=95, b=45, l=70, r=30), updatemenus=[dict(buttons=botones, x=0, y=1.15)],
              hovermode="closest", legend=dict(title=dict(text="Municipio")))
salida = figuras / "eva_Py_09_series_interactivas.html"

biblioteca = Path("recursos") / "plotly-2.35.2.min.js"
if not (figuras / biblioteca).is_file():
    raise FileNotFoundError(f"Falta la biblioteca incluida en la entrega: {figuras / biblioteca}")
datos_js = json.dumps(trazas, allow_nan=False).replace("<", r"\u003c")
opciones_js = json.dumps(diseno).replace("<", r"\u003c")
contenido = ("<!doctype html><html lang='es'><head><meta charset='utf-8'>"
             "<title>Rendimiento municipal · EVA Cesar</title>"
             f"<script src='{biblioteca.as_posix()}'></script></head>"
             "<body><main id='serie' style='height:94vh'></main><script>"
             "if (typeof Plotly === 'undefined') {"
             "document.querySelector('#serie').textContent = 'Comprueba la carpeta recursos de la entrega.';"
             "} else {"
             f"Plotly.newPlot('serie', {datos_js}, "
             f"{opciones_js}, {{responsive:true,displaylogo:false}});"
             "}</script></body></html>")
salida.write_text(contenido, encoding="utf-8")
print(f"Gráfico Plotly: {salida.relative_to(raiz)}")
